# NB4 — Stage B: learn skill B four ways

For the next unfinished seed, starting from its Stage-A model: ordinary (sequential), protected, random-Q and replay DoRA on skill B — two methods per GPU, same seed and token budget — each dev-selected and merged into a verified patch.

**Accelerator:** GPU T4 x2 · **Internet:** OFF · **Attach:** code, model, NB2, all NB3 outputs (+ every earlier NB4 output)

Run with **Save Version → Save & Run All (Commit)**. Afterwards: run it again (attaching this output too) until it prints "Stage B finished for all seeds".

In [ ]:
SESSION_HOURS = 12   # your GPU session limit (see the notebook settings panel)
# --- Setup (identical in every notebook; nothing to edit) --------------------
import json, os, shutil, subprocess, sys, time
from pathlib import Path
NOTEBOOK_STARTED = time.time()
KAGGLE = Path(os.environ.get('PCL_KAGGLE_ROOT', '/kaggle'))   # override only for a local rehearsal
INPUT, WORKING = KAGGLE / 'input', KAGGLE / 'working'

code_roots = sorted({p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'pcl' / 'cli.py').is_file()})
assert len(code_roots) == 1, f'attach exactly one copy of the pcl code dataset; found {code_roots}'
WORK = Path(os.environ.get('PCL_WORK', '/tmp/protected-cl'))
if not WORK.exists():
    shutil.copytree(code_roots[0], WORK, ignore=shutil.ignore_patterns('outputs', '__pycache__', '*.egg-info', '.pytest_cache'))
sys.path.insert(0, str(WORK))   # `python -m pcl` runs from WORK, so no package install is needed

import importlib.metadata as metadata
def _version(name):
    try:
        return tuple(int(x) for x in metadata.version(name).split('.')[:2])
    except Exception:
        return (0,)
needed = {'transformers': (4, 51), 'safetensors': (0, 4), 'accelerate': (1, 0), 'PyYAML': (6, 0)}
missing = [f'{n}>={".".join(map(str, v))}' for n, v in needed.items() if _version(n) < v]
if missing:
    print('Installing', missing, '(needs Internet ON for this notebook)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import yaml
from pcl import kaggle
from pcl.io import read_json

def pcl(*args):
    """Run one pcl command in a fresh process (its GPU memory is released when it exits)."""
    print('$ pcl', ' '.join(map(str, args)), flush=True)
    subprocess.run([sys.executable, '-m', 'pcl', *map(str, args)], check=True, cwd=WORK)

def on_gpus(commands, log_dir):
    """One pcl command per T4, logs streamed live; fails if either command fails."""
    commands = [json.dumps([sys.executable, '-m', 'pcl', *map(str, c)]) for c in commands]
    subprocess.run([sys.executable, str(WORK / 'scripts' / 'run_pair.py'), '--out', str(log_dir), *commands],
                   check=True, cwd=WORK)

def show(path):
    try:
        from IPython.display import Markdown, display
        display(Markdown(Path(path).read_text()))
    except ImportError:
        print(Path(path).read_text())

def remaining_seconds():
    """Session time left, from SESSION_HOURS in the first cell."""
    return int(SESSION_HOURS * 3600 - (time.time() - NOTEBOOK_STARTED))

found = kaggle.discover(INPUT, exclude=code_roots[0])
print({key: len(value) for key, value in found.items()})

In [ ]:
MODEL = kaggle.one(found, 'models', 'pinned model Dataset (made from the NB1 output)')
M0_CONFIG = kaggle.m0_config(MODEL, WORKING / 'configs' / 'm0.yaml')
DATA = kaggle.one(found, 'data', 'NB2 output (data)')
PILOT = kaggle.one(found, 'pilot', 'NB2 output (pilot)')
SETTINGS = kaggle.study_settings(PILOT, WORKING / 'study.yaml')
settings = yaml.safe_load(SETTINGS.read_text())
assert settings['precision_validated'], 'The pilot did not validate FP16. Read pilot/pilot_report.md first.'
print(settings)

In [ ]:
# Finish the lowest unfinished seed, then keep going with the next seed while at least
# MIN_HOURS_FOR_NEXT_SEED of the session remain (a seed cut off by the clock resumes next run).
MIN_HOURS_FOR_NEXT_SEED = 2
everything, attempted = found, set()
SEED = kaggle.plan_stage_b(everything)
while SEED is not None and SEED not in attempted and remaining_seconds() > MIN_HOURS_FOR_NEXT_SEED * 3600:
    attempted.add(SEED)
    STAGE_A = kaggle.stage_a_for(found, SEED)
    previous = kaggle.stage_dirs(found, 'b', SEED)
    commands = [['stage-b', '--model-config', M0_CONFIG, '--settings', SETTINGS, '--data-root', DATA, '--seed', SEED,
                 '--stage-a', STAGE_A, '--methods', *methods, '--out', WORKING / 'stage_b' / f'seed{SEED}' / f'gpu{gpu}',
                 '--session-seconds', remaining_seconds(), *(['--previous', *previous] if previous else [])]
                for gpu, methods in enumerate([['sequential', 'replay'], ['protected', 'random_q']])]
    on_gpus(commands, WORKING / 'logs' / f'seed{SEED}')
    everything = {'stages': found['stages'] + kaggle.discover(WORKING)['stages']}
    print(f'seed {SEED}: finished', sorted(kaggle.stage_b_done(everything, SEED)))
    SEED = kaggle.plan_stage_b(everything)
if SEED is None:
    print('Stage B finished for all seeds. Next: NB5.')
else:
    print(f'Next NB4 run continues with seed {SEED} (attach this output as a Dataset).')